# Experiment Summary: YOLOv8 Baseline vs. YOLOv8 Reasoner for Thermal Object Detection

This notebook documents an experiment to compare the performance of a standard YOLOv8 baseline model against a custom YOLOv8 Reasoner model for object detection on thermal imagery from the FLIR dataset.

## Experiment Steps:

1.  **Environment Setup**: Google Drive was mounted, the working directory was set to the `ultralytics-main` repository, and the necessary `ultralytics` package was installed from the local repository.

2.  **Reasoner Integration Validation**: The custom YOLOv8 Reasoner model architecture was loaded and its structure printed to confirm successful integration.

3.  **Inference Test (Single Image)**: Initial inference was performed on a single test image (`bus.jpg`) using both the baseline (untrained) and Reasoner (untrained) models to verify their functionality. Results were saved for visual inspection.

4.  **Model Training (Attempted)**: Attempts were made to train both the Baseline YOLOv8 and YOLOv8 Reasoner models using the FLIR thermal dataset and subsequent steps indicate models were loaded, implying an alternative training method was successful.

5.  **Batch Inference & Result Stitching**: Both trained baseline and reasoner models were used to perform inference on a batch of test images from the FLIR dataset. The original image, baseline model predictions, and reasoner model predictions were then stitched together horizontally for direct visual comparison and saved as `stitched_*.jpg` files in the `runs/stitched_results` directory.

6.  **Model Evaluation**: Performance metrics, including Precision, Recall, mAP50, and mAP50-95, were calculated for both the baseline and reasoner models using the `YOLO.val()` method on the FLIR validation set. These metrics were then compiled into a pandas DataFrame and saved to an Excel file (`yolov8_model_evaluation_metrics.xlsx`) for quantitative comparison.

In [1]:
#01.Environment Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
#01.Environment Setup
import os

# Define the path to your working directory within Google Drive (repository root)
working_dir = '/content/drive/My Drive/YOLOV8-IR/ultralytics-main'

try:
    # Change the current working directory to the specified path
    os.chdir(working_dir)
    print(f"Current working directory changed to: {os.getcwd()}")

    # List the contents of the directory to confirm it's correct
    print("Contents of the directory:")
    for item in os.listdir('.'):
        print(item)
except FileNotFoundError:
    print(f"Error: The directory '{working_dir}' was not found.")
    print("Please check the path. Here are the contents of your 'My Drive' directory:")
    my_drive_path = '/content/drive/My Drive'
    if os.path.exists(my_drive_path):
        for item in os.listdir(my_drive_path):
            print(os.path.join(my_drive_path, item))
    else:
        print("My Drive is not mounted or accessible at /content/drive/My Drive.")
    print("You might need to correct the 'working_dir' variable or ensure your Drive is mounted correctly.")

Current working directory changed to: /content/drive/My Drive/YOLOV8-IR/ultralytics-main
Contents of the directory:
.dockerignore
README.zh-CN.md
.gitignore
README.md
CITATION.cff
CONTRIBUTING.md
LICENSE
pyproject.toml
mkdocs.yml
.github
examples
tests
docs
docker
ultralytics
ultralytics.egg-info
runs
yolov8s.pt
yolo26n.pt
datasets


In [4]:
#01.Environment Setup
!pip install e .

Processing /content/drive/MyDrive/YOLOV8-IR/ultralytics-main
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for e: filename=e-1.4.5-py3-none-any.whl size=2793 sha256=09a92a78dccb9c79af792bf606ea8c049f0075aabfecf79a8ce8bddd8ad95ad7
  Stored in directory: /root/.cache/pip/wheels/d8/09/f1/8408d97979e461fd397abc90a6f1a6a5b00f40fcf4f1c8b2ce
  Created wheel for ultralytics: filename=ultralytics-8.4.41-py3-none-any.whl size=1230986 sha256=a05219435e11c359aca5957e102b8ea95ef7575e7aec57dd33d7d10ecfa24874
  Stored in directory: /root/.cache/pip/wheels/80/5c/e0/2d5b99efde2651e04f6b4e6471d946c4dc4b44f733cccbce6d
Successfully built e ultralytics


In [5]:
#02. Validate reasoner integration
from ultralytics import YOLO

# Load a model
model = YOLO('ultralytics/cfg/models/v8/yolov8_reasoner.yaml')
print(model.model)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ no model scale passed. Assuming scale='n'.
Reasoner forward: torch.Size([1, 64, 32, 32])
Reasoner forward: torch.Size([1, 128, 16, 16])
Reasoner forward: torch.Size([1, 256, 8, 8])
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inp

In [ ]:
#03.Inference Test (Single Image)

from ultralytics import YOLO

# Load the model
model = YOLO('ultralytics/cfg/models/v8/yolov8_reasoner.yaml')

# Perform inference using the model object
results = model.predict(source='ultralytics/assets/bus.jpg', imgsz=640, save=True, project='flir_experiments', name='yolov8_reasoner_inference')

print("Model is untrained and no detection is expected. Results saved to the 'flir_experiments/yolov8_reasoner_inference' directory.")

In [ ]:
#04. Train the Baseline YOLOv8

!yolo detect train \
  model=ultralytics/cfg/models/v8/yolov8.yaml \
  scale = s \
  data=/content/drive/MyDrive/YOLOV8-IR/FLIR_Datset/flir_thermal_data.yaml \
  epochs=100 \
  imgsz=640 \
  batch=16 \
  seed=42 \
  project = flir_experiments \
  name = yolov8_baseline \
  lr0=0.003 \
  hsv_h=0.0 \
  hsv_s=0.0 \
  hsv_v=0.0 \
  degrees=0.0 \
  translate=0.1 \
  scale=0.5 \
  shear=0.0 \
  fliplr=0.5 \
  mosaic=1.0 \
  mixup=0.1

/bin/bash: line 1: yolo: command not found


In [ ]:
#04. Train the YOLOv8 Reasoner model

!yolo detect train \
  model=ultralytics/cfg/models/v8/yolov8_reasoner.yaml \
  scale=s \
  data='/content/drive/MyDrive/YOLOV8-IR/FLIR_Datset/flir_thermal_data.yaml' \
  epochs=100 \
  imgsz=640 \
  batch=16 \
  seed=42 \
  project=flir_experiments \
  name=yolov8_reasoner-7 \
  model = /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_reasoner-7/weights/last.pt \
  resume = True \
  lr0=0.003 \
  hsv_h=0.0 \
  hsv_s=0.0 \
  hsv_v=0.0 \
  degrees=0.0 \
  translate=0.1 \
  scale=0.5 \
  shear=0.0 \
  fliplr=0.5 \
  mosaic=1.0 \
  mixup=0.1

In [8]:
#05. Batch Inference & Result Stitching

from ultralytics import YOLO
import os

# --- Baseline YOLOv8 Inference ---
print("\n--- Performing inference for Baseline YOLOv8 ---")
# Load the baseline model (assuming yolov8s.pt is available or train it first)
# If you trained yolov8_baseline, you might want to load its best.pt or last.pt
baseline_model_path = '/content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_baseline/weights/best.pt' # Or your trained model like 'runs/detect/flir_experiments/yolov8_baseline/weights/best.pt'
baseline_model = YOLO(baseline_model_path)

# Perform inference for the baseline model
baseline_results = baseline_model.predict(
    source='ultralytics/assets/bus.jpg',
    imgsz=640,
    save=True,
    project='flir_inference_results',
    name='yolov8_baseline_inference'
)
print(f"Baseline model inference results saved to: {os.path.join('flir_inference_results', 'yolov8_baseline_inference')}")

# --- YOLOv8 Reasoner Model Inference ---
print("\n--- Performing inference for YOLOv8 Reasoner model ---")
# Load the reasoner model
# If you trained yolov8_reasoner-7, you might want to load its best.pt or last.pt
reasoner_model_path = '/content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_reasoner-7/weights/best.pt' # Or your trained model like 'runs/detect/flir_experiments/yolov8_reasoner-7/weights/best.pt'
reasoner_model = YOLO(reasoner_model_path)

# Perform inference for the reasoner model
reasoner_results = reasoner_model.predict(
    source='ultralytics/assets/bus.jpg',
    imgsz=640,
    save=True,
    project='flir_inference_results',
    name='yolov8_reasoner_inference'
)
print(f"Reasoner model inference results saved to: {os.path.join('flir_inference_results', 'yolov8_reasoner_inference')}")

print("\nInference complete for both models.")



--- Performing inference for Baseline YOLOv8 ---

image 1/1 /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/ultralytics/assets/bus.jpg: 640x480 (no detections), 9.5ms
Speed: 2.6ms preprocess, 9.5ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_inference_results/yolov8_baseline_inference-2
Baseline model inference results saved to: flir_inference_results/yolov8_baseline_inference

--- Performing inference for YOLOv8 Reasoner model ---

Reasoner forward: torch.Size([1, 64, 80, 80])
Reasoner forward: torch.Size([1, 128, 40, 40])
Reasoner forward: torch.Size([1, 256, 20, 20])
Reasoner forward: torch.Size([1, 64, 80, 60])
Reasoner forward: torch.Size([1, 128, 40, 30])
Reasoner forward: torch.Size([1, 256, 20, 15])
image 1/1 /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/ultralytics/assets/bus.jpg: 640x480 (no detections), 24.8ms
Speed: 2.2ms preprocess, 24.8ms inference, 0.6ms postpr

In [9]:
# 06. Model Evaluation
import os
import cv2
from ultralytics import YOLO
import numpy as np

# Define paths
test_images_dir = '/content/drive/MyDrive/YOLOV8-IR/FLIR_Datset/test_images'
output_stitched_dir = os.path.join(os.getcwd(), 'runs/stitched_results')
os.makedirs(output_stitched_dir, exist_ok=True)

# Model paths from previously trained models (as per your kernel state)
baseline_model_path = '/content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_baseline/weights/best.pt'
reasoner_model_path = '/content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_reasoner-7/weights/best.pt'

# Load models
print(f"Loading baseline model from: {baseline_model_path}")
baseline_model = YOLO(baseline_model_path)
print(f"Loading reasoner model from: {reasoner_model_path}")
reasoner_model = YOLO(reasoner_model_path)

# Get list of images
image_files = [f for f in os.listdir(test_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Found {len(image_files)} images in {test_images_dir}")
print(f"Stitched results will be saved to {output_stitched_dir}")

for img_file in image_files:
    source_image_path = os.path.join(test_images_dir, img_file)
    image_name_without_ext = os.path.splitext(img_file)[0]

    print(f"\nProcessing {img_file}...")

    # 1. Inference for Baseline Model
    print("Performing baseline inference...")
    baseline_inference_results = baseline_model.predict(
        source=source_image_path,
        imgsz=640,
        save=True,
        project='flir_inference_results_batch',
        name='baseline_batch_inference',
        exist_ok=True # Overwrite existing results for same image file
    )
    # Construct path to the saved baseline result image
    # Ultralytics saves the processed image in a subfolder named by the original filename without extension (sometimes, depending on version)
    # or directly using the original filename. We need to be robust.
    baseline_result_dir = os.path.join(os.getcwd(), 'runs/detect/flir_inference_results_batch/baseline_batch_inference')
    baseline_result_path = os.path.join(baseline_result_dir, img_file)
    # Check if a subfolder was created (e.g., if multiple images have same name across different paths)
    if not os.path.exists(baseline_result_path):
      possible_subfolder = os.path.join(baseline_result_dir, image_name_without_ext, img_file)
      if os.path.exists(possible_subfolder):
        baseline_result_path = possible_subfolder
      else:
        print(f"Warning: Could not find baseline result for {img_file} at {baseline_result_path} or {possible_subfolder}. Skipping stitching for this image.")
        continue

    # 2. Inference for Reasoner Model
    print("Performing reasoner inference...")
    reasoner_inference_results = reasoner_model.predict(
        source=source_image_path,
        imgsz=640,
        save=True,
        project='flir_inference_results_batch',
        name='reasoner_batch_inference',
        exist_ok=True # Overwrite existing results for same image file
    )
    # Construct path to the saved reasoner result image
    reasoner_result_dir = os.path.join(os.getcwd(), 'runs/detect/flir_inference_results_batch/reasoner_batch_inference')
    reasoner_result_path = os.path.join(reasoner_result_dir, img_file)
    if not os.path.exists(reasoner_result_path):
      possible_subfolder = os.path.join(reasoner_result_dir, image_name_without_ext, img_file)
      if os.path.exists(possible_subfolder):
        reasoner_result_path = possible_subfolder
      else:
        print(f"Warning: Could not find reasoner result for {img_file} at {reasoner_result_path} or {possible_subfolder}. Skipping stitching for this image.")
        continue

    # 3. Stitch images
    try:
        original_img = cv2.imread(source_image_path)
        baseline_res_img = cv2.imread(baseline_result_path)
        reasoner_res_img = cv2.imread(reasoner_result_path)

        if original_img is None or baseline_res_img is None or reasoner_res_img is None:
            print(f"Error: Failed to load one or more images for {img_file}. Skipping stitching.")
            continue

        # Resize all images to a common height for uniform stitching
        target_height = original_img.shape[0]

        baseline_res_img = cv2.resize(baseline_res_img, (int(baseline_res_img.shape[1] * (target_height / baseline_res_img.shape[0])), target_height))
        reasoner_res_img = cv2.resize(reasoner_res_img, (int(reasoner_res_img.shape[1] * (target_height / reasoner_res_img.shape[0])), target_height))

        # Add titles to images
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1
        font_thickness = 2
        text_color = (0, 0, 255) # Red color for text (BGR format)
        text_y_offset = 30 # Y position for text
        padding = 10 # Padding for text background

        def add_title_with_bg(img, title, color=(0, 0, 0)): # Black background
            img_with_title = img.copy()
            (text_width, text_height), baseline = cv2.getTextSize(title, font, font_scale, font_thickness)
            text_x = (img_with_title.shape[1] - text_width) // 2
            text_y = text_y_offset

            # Draw a filled rectangle as background for the text
            cv2.rectangle(img_with_title, (text_x - padding, text_y - text_height - padding),
                          (text_x + text_width + padding, text_y + baseline + padding), color, -1)
            cv2.putText(img_with_title, title, (text_x, text_y), font, font_scale, text_color, font_thickness, cv2.LINE_AA)
            return img_with_title

        original_img_titled = add_title_with_bg(original_img, "Original Image", (255, 255, 255)) # White background for original
        baseline_res_img_titled = add_title_with_bg(baseline_res_img, "Baseline YOLOv8", (255, 255, 255))
        reasoner_res_img_titled = add_title_with_bg(reasoner_res_img, "YOLOv8 Reasoner", (255, 255, 255))

        # Ensure all images have the same number of channels (e.g., all 3-channel for color)
        if len(original_img_titled.shape) == 2:
            original_img_titled = cv2.cvtColor(original_img_titled, cv2.COLOR_GRAY2BGR)
        if len(baseline_res_img_titled.shape) == 2:
            baseline_res_img_titled = cv2.cvtColor(baseline_res_img_titled, cv2.COLOR_GRAY2BGR)
        if len(reasoner_res_img_titled.shape) == 2:
            reasoner_res_img_titled = cv2.cvtColor(reasoner_res_img_titled, cv2.COLOR_GRAY2BGR)

        # Stitch horizontally
        stitched_image = cv2.hconcat([original_img_titled, baseline_res_img_titled, reasoner_res_img_titled])

        # Save the stitched image
        stitched_output_path = os.path.join(output_stitched_dir, f'stitched_{img_file}')
        cv2.imwrite(stitched_output_path, stitched_image)
        print(f"Saved stitched image to {stitched_output_path}")

    except Exception as e:
        print(f"Error stitching images for {img_file}: {e}")

print("\nAll images processed and stitched.")

Loading baseline model from: /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_baseline/weights/best.pt
Loading reasoner model from: /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_reasoner-7/weights/best.pt
Found 20 images in /content/drive/MyDrive/YOLOV8-IR/FLIR_Datset/test_images
Stitched results will be saved to /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/stitched_results

Processing video-4FRnNpmSmwktFJKjg-frame-000981-DpA4nTkLXA7ERAAy2.jpg...
Performing baseline inference...

image 1/1 /content/drive/MyDrive/YOLOV8-IR/FLIR_Datset/test_images/video-4FRnNpmSmwktFJKjg-frame-000981-DpA4nTkLXA7ERAAy2.jpg: 512x640 2 persons, 42.1ms
Speed: 1.6ms preprocess, 42.1ms inference, 1.3ms postprocess per image at shape (1, 3, 512, 640)
Results saved to /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_inference_results_batch/baseline_batch_inference
Performing reasoner inference...

Reasoner fo

In [12]:
# 06. Model Evaluation Metrics
from ultralytics import YOLO
import os

# Define paths to your trained models and data configuration
baseline_model_path = '/content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_baseline/weights/best.pt'
reasoner_model_path = '/content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/detect/flir_experiments/yolov8_reasoner-7-2/weights/best.pt'
data_yaml_path = '/content/drive/My Drive/YOLOV8-IR/FLIR_Datset/flir_thermal_data.yaml'

# --- Evaluate Baseline YOLOv8 Model ---
print("\n--- Evaluating Baseline YOLOv8 Model ---")
if os.path.exists(baseline_model_path):
    baseline_model = YOLO(baseline_model_path)
    baseline_metrics = baseline_model.val(
        data=data_yaml_path,
        imgsz=640,
        batch=16,
        project='flir_evaluation_results',
        name='yolov8_baseline_metrics',
        exist_ok=True # Overwrite existing results
    )

    print("\nBaseline Model Metrics:")
    print(f"  Precision (P): {baseline_metrics.results_dict['metrics/precision(B)']:.3f}")
    print(f"  Recall (R): {baseline_metrics.results_dict['metrics/recall(B)']:.3f}")
    print(f"  mAP50: {baseline_metrics.results_dict['metrics/mAP50(B)']:.3f}")
    print(f"  mAP50-95: {baseline_metrics.results_dict['metrics/mAP50-95(B)']:.3f}")
    # F1 score is not directly reported in results_dict, often calculated as a harmonic mean of P and R
    # For a rough estimate, you can calculate it, but it's more complex with varying confidence thresholds
    # print(f"  F1 Score (approx): {2 * (baseline_metrics.results_dict['metrics/precision(B)'] * baseline_metrics.results_dict['metrics/recall(B)']) / (baseline_metrics.results_dict['metrics/precision(B)'] + baseline_metrics.results_dict['metrics/recall(B)']):.3f}")
    print(f"Results saved to: {os.path.join('flir_evaluation_results', 'yolov8_baseline_metrics')}")
else:
    print(f"Warning: Baseline model not found at {baseline_model_path}. Please ensure it is trained.")

# --- Evaluate YOLOv8 Reasoner Model ---
print("\n--- Evaluating YOLOv8 Reasoner Model ---")
if os.path.exists(reasoner_model_path):
    reasoner_model = YOLO(reasoner_model_path)
    reasoner_metrics = reasoner_model.val(
        data=data_yaml_path,
        imgsz=640,
        batch=16,
        project='flir_evaluation_results',
        name='yolov8_reasoner_metrics',
        exist_ok=True # Overwrite existing results
    )

    print("\nReasoner Model Metrics:")
    print(f"  Precision (P): {reasoner_metrics.results_dict['metrics/precision(B)']:.3f}")
    print(f"  Recall (R): {reasoner_metrics.results_dict['metrics/recall(B)']:.3f}")
    print(f"  mAP50: {reasoner_metrics.results_dict['metrics/mAP50(B)']:.3f}")
    print(f"  mAP50-95: {reasoner_metrics.results_dict['metrics/mAP50-95(B)']:.3f}")
    # print(f"  F1 Score (approx): {2 * (reasoner_metrics.results_dict['metrics/precision(B)'] * reasoner_metrics.results_dict['metrics/recall(B)']) / (reasoner_metrics.results_dict['metrics/precision(B)'] + reasoner_metrics.results_dict['metrics/recall(B)']):.3f}")
    print(f"Results saved to: {os.path.join('flir_evaluation_results', 'yolov8_reasoner_metrics')}")
else:
    print(f"Warning: Reasoner model not found at {reasoner_model_path}. Please ensure it is trained.")

print("\nEvaluation complete for both models.")


--- Evaluating Baseline YOLOv8 Model ---
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8 summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 1.6±2.4 ms, read: 39.7±28.6 MB/s, size: 94.8 KB)
val: Scanning /content/drive/MyDrive/YOLOV8-IR/FLIR_Datset/labels/val.cache... 245 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 250/250 55.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.4it/s 6.7s
                   all        250       3799      0.353      0.234      0.206     0.0991
                person        189       1050      0.368      0.608      0.499      0.213
                  bike         32         38     0.0757      0.263      0.151     0.0643
                   car        225       1570      0.417       0.66      0.621      0.357
            motorcycle          9         10      0.694        0.4      

In [13]:
# 06. Model Evaluation Metrics consolidated
import pandas as pd
import os

# Extract metrics from the previous cell's output (assuming the output is consistent)
# This is a manual extraction based on the provided output structure.

baseline_metrics_data = {
    'Model': 'YOLOv8 Baseline',
    'Precision (P)': 0.353,
    'Recall (R)': 0.234,
    'mAP50': 0.206,
    'mAP50-95': 0.099
}

reasoner_metrics_data = {
    'Model': 'YOLOv8 Reasoner',
    'Precision (P)': 0.388,
    'Recall (R)': 0.231,
    'mAP50': 0.213,
    'mAP50-95': 0.105
}

# Create DataFrame
metrics_df = pd.DataFrame([baseline_metrics_data, reasoner_metrics_data])

# Define output path
output_dir = os.path.join(os.getcwd(), 'runs/evaluation_results')
os.makedirs(output_dir, exist_ok=True)
excel_output_path = os.path.join(output_dir, 'yolov8_model_evaluation_metrics.xlsx')

# Save to Excel
metrics_df.to_excel(excel_output_path, index=False)

print(f"Evaluation metrics saved to: {excel_output_path}")
display(metrics_df)

Evaluation metrics saved to: /content/drive/MyDrive/YOLOV8-IR/ultralytics-main/runs/evaluation_results/yolov8_model_evaluation_metrics.xlsx


,Model,Precision (P),Recall (R),mAP50,mAP50-95
0,YOLOv8 Baseline,0.353,0.234,0.206,0.099
1,YOLOv8 Reasoner,0.388,0.231,0.213,0.105
